In [2]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip all RAVDESS video files into Colab
import os

zip_dir = "/content/drive/MyDrive/RAVDESS"
output_dir = "/content/RAVDESS"

os.makedirs(output_dir, exist_ok=True)

for zip_file in os.listdir(zip_dir):
    if zip_file.endswith(".zip"):
        zip_path = os.path.join(zip_dir, zip_file)
        !unzip -q "{zip_path}" -d "{output_dir}"

print("✅ All RAVDESS video files extracted!")

# 3. Extract frames from videos (using OpenCV)
import cv2

frame_dir = "/content/RAVDESS_frames"
os.makedirs(frame_dir, exist_ok=True)

for root, dirs, files in os.walk(output_dir):
    for file in files:
        if file.endswith(".mp4"):
            video_path = os.path.join(root, file)
            cap = cv2.VideoCapture(video_path)
            count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                # Save 1 frame per second
                if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % int(cap.get(cv2.CAP_PROP_FPS)) == 0:
                    frame_name = f"{os.path.splitext(file)[0]}_{count}.jpg"
                    cv2.imwrite(os.path.join(frame_dir, frame_name), frame)
                    count += 1
            cap.release()

print("✅ Frames extracted from RAVDESS videos!")




Mounted at /content/drive
✅ All RAVDESS video files extracted!
✅ Frames extracted from RAVDESS videos!


In [3]:
# 4. Training setup with PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

class RAVDESSFrames(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)]
        # Example: binary labels (happy vs not happy)
        self.labels = [0 if "happy" not in f else 1 for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

train_dataset = RAVDESSFrames(frame_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 5. Load ResNet18
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # Binary classification
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 6. Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

    # Save checkpoint to Drive
    torch.save(model.state_dict(), f"/content/drive/MyDrive/RAVDESS_checkpoint_epoch{epoch+1}.pth")

print("🎉 Training finished!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 87.7MB/s]


Epoch 1/10, Loss: 0.0129, Accuracy: 99.85%
Epoch 2/10, Loss: 0.0004, Accuracy: 100.00%
Epoch 3/10, Loss: 0.0002, Accuracy: 100.00%
Epoch 4/10, Loss: 0.0001, Accuracy: 100.00%
Epoch 5/10, Loss: 0.0001, Accuracy: 100.00%
Epoch 6/10, Loss: 0.0001, Accuracy: 100.00%
Epoch 7/10, Loss: 0.0001, Accuracy: 100.00%
Epoch 8/10, Loss: 0.0000, Accuracy: 100.00%
Epoch 9/10, Loss: 0.0000, Accuracy: 100.00%
Epoch 10/10, Loss: 0.0000, Accuracy: 100.00%
🎉 Training finished!


In [5]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

# Dataset class: sample one frame per video
class CREMAVideoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.video_files = [os.path.join(root_dir, f)
                            for f in os.listdir(root_dir) if f.endswith(".flv")]

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = self.video_files[idx]
        label = 1 if "HAP" in video_path else 0  # happy vs not happy

        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        target_frame = frame_count // 2  # middle frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
        ret, frame = cap.read()
        cap.release()

        # Convert to PIL Image
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if self.transform:
            image = self.transform(image)
        return image, label

# Transform (same as training)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# Validation dataset and loader
crema_video_dir = "/content/drive/MyDrive/VideoFlash"
val_dataset = CREMAVideoDataset(crema_video_dir, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Load trained model checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("/content/drive/MyDrive/RAVDESS_checkpoint_epoch10.pth"))
model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

# Validation loop
val_loss, val_correct, val_total = 0, 0, 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

print(f"Validation Loss: {val_loss/len(val_loader):.4f}, "
      f"Validation Accuracy: {100*val_correct/val_total:.2f}%")


Validation Loss: 1.0370, Validation Accuracy: 82.93%


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

# Dataset class: sample one frame per video
class AFEWVideoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.video_files = [os.path.join(root_dir, f)
                            for f in os.listdir(root_dir) if f.endswith(".avi") or f.endswith(".mp4")]

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = self.video_files[idx]

        # Example: parse label from filename (depends on AFEW naming convention)
        # Adjust this line based on how your AFEW files are named
        if "Happy" in video_path:
            label = 1
        else:
            label = 0  # not happy

        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        target_frame = frame_count // 2  # middle frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
        ret, frame = cap.read()
        cap.release()

        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if self.transform:
            image = self.transform(image)
        return image, label

# Transform (same as training)
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# Test dataset and loader
afew_video_dir = "/content/drive/MyDrive/AFEW/VideoClips"
test_dataset = AFEWVideoDataset(afew_video_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load trained model checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("/content/drive/MyDrive/RAVDESS_checkpoint_epoch10.pth"))
model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

# Test loop
test_loss, test_correct, test_total = 0, 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"Test Loss: {test_loss/len(test_loader):.4f}, "
      f"Test Accuracy: {100*test_correct/test_total:.2f}%")
